# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [ ]:
VARIANT = None   # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [ ]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [ ]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

#### ✍️ Дизайн-документ

Заполните правую колонку для своей акции — как в разделе 8 семинара С7 и на слайде 32 лекции Л7. Многоточия замените своим текстом.

| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | Акция «Второй месяц Расширенного за 1 ₽» должна увеличить оборот в партнерских сервисах за 4 недели. Бизнес-ориентир - рост минимум на 3%, или примерно 94 ₽ на клиента |
| единица рандомизации и почему именно она (подпиской делятся — колонка `members`) | Аккаунт клиента (client_id), а не отдельный участник общей подписки. Учитываем members: несколько людей могут пользоваться одной подпиской |
| группы и доли | План 50/50: контроль - обычные условия, тест - предложение акции, по 3000 клиентов на группу |
| основная метрика | spend - средний оборот в партнерских сервисах за 4 недели на клиента, ₽ |
| метрики-стражи: что не должно ухудшиться | churned — отток клиентов, tickets — число обращений в поддержку |
| α, мощность, MDE — числами из ячейки выше | α = 0,05, мощность = 80%, MDE без CUPED = 230 ₽, с CUPED = 186 ₽ |
| поправка на множественные сравнения | Метод Холма для проверки нескольких метрик |
| оценщик | Разность средних test − control, 95%-й доверительный интервал, CUPED с ковариатой spend_prev |
| окно | 4 недели эксперимента |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | Запускаем, если нижняя граница 95% доверительного интервала выше целевого прироста 94 ₽, стражи не ухудшились и акция окупается с учетом стоимости скидки. Дорабатываем и повторяем тест, если эффект не подтвержден или экономическая выгода пока неясна. Отказываемся, если акция приводит к существенному ухудшению стражей или остается убыточной после доработки |

## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes)

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
effect = diff_ci(test["spend"], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [ ]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [ ]:
# Числа для ответа на вопрос 1 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.

#собираем результаты задач 1-4 для ответа
print(f"SRM: p = {srm.pvalue:.4f}")
print(f"Баланс до эксперимента: p = {balance.pvalue:.4f}")
print(f"Эффект: {effect['diff']:.2f} ₽")
print(f"95% ДИ: [{effect['lo']:.2f}; {effect['hi']:.2f}] ₽")
print(f"MDE без CUPED: {mde_raw:.0f} ₽, с CUPED: {mde_cuped:.0f} ₽")
print(f"t-test: p = {ab['TTest p-value'].iloc[0]:.4f}")
print(f"U-test: p = {ab['UTest p-value'].iloc[0]:.4f}")

#### ✍️ Ответ на вопрос 1

_Разбиению можно предварительно доверять: SRM: p = 0.3394 и проверка баланса до эксперимента: p = 0.7876 - оба значения больше 0,05, поэтому статистически значимых отклонений не найдено_

_Эффект положительный, но незначимый: +117.48 ₽, 95% ДИ [-55.82; 290.79] ₽ - интервал включает ноль, p = 0.1840. Наблюдаемая разность меньше MDE 230 ₽ (и расчетного MDE с CUPED 186 ₽)_

_t-test и U-test не противоречат друг другу: p = 0.1840 и 0.4106 - оба не обнаружили значимых различий, а p-value отличаются, потому что t-критерий сравнивает средние, а U-критерий основан на рангах наблюдений_

## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [ ]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table).pvalue

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [ ]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [ ]:
# Ваша cuped() из pilot.py: метрика spend, ковариата spend_prev -> mine["spend_cuped"] и theta.
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [ ]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole(parent="spend", lag=1)

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [ ]:
# Числа для ответа на вопрос 2 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.


#проверяем сработали ли метрики стражи
print("Отток: разница =", round(guards.loc["churned", "diff"] * 100, 2), "п.п.")
print("Отток: p =", round(p_churn, 4))
print("Обращения: p =", round(guards.loc["tickets", "p"], 4))
print("Значимые метрики после Холма:", answer_6)

#сравниваем ширину доверительных интервалов до и после CUPED
width_before = effect["hi"] - effect["lo"]
width_after = effect_cuped["hi"] - effect_cuped["lo"]
narrowing = (1 - width_after / width_before) * 100

print(f"Интервал сузился на {narrowing:.1f}%")
print(f"Эффект до CUPED: {effect['diff']:.2f} ₽")
print(f"Эффект после CUPED: {effect_cuped['diff']:.2f} ₽")

#проверяем различия между группами до эксперимента
pre_diff = test["spend_prev"].mean() - control["spend_prev"].mean()
print(f"Прошлый оборот test: {test['spend_prev'].mean():.2f} ₽")
print(f"Прошлый оборот control: {control['spend_prev'].mean():.2f} ₽")
print(f"Разница до эксперимента: {pre_diff:.2f} ₽")

#### ✍️ Ответ на вопрос 2

_Значимого ухудшения метрик-стражей не обнаружено: отток вырос на 0.08 п.п. (p = 0.9211), а обращения на 0.0014 на клиента (p = 0.8247). После поправки Холма ни одна из 4 метрик не осталась статистически значимой: CUPED сузил доверительный интервал на 20.9% и немного увеличил оценку эффекта- с 117.48 ₽ до 132.04 ₽, тк до эксперимента тестовая группа тратила в среднем на 22.13 ₽ меньше контрольной (3127.64 ₽ против 3149.77 ₽). При этом интервал после CUPED все еще включает ноль (от −5.03 до 269.11 ₽), поэтому статистически значимый рост оборота пока не подтвержден_

## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [ ]:
# Задача 9. 95 % интервал разности средних оборота test − control бутстрепом:
# 2 000 повторов, rng = np.random.default_rng(VARIANT), сначала тест, потом контроль.

#берем обороты клиентов тестовой и контрольной групп
x = test["spend"].to_numpy()
y = control["spend"].to_numpy()
#фиксируем случайность для воспроизводимости результата
rng = np.random.default_rng(VARIANT)
#2000 раз формируем новые выборки и считаем разность средних
diffs = []
for _ in range(2000):
    test_sample = rng.choice(x, size=len(x), replace=True)
    control_sample = rng.choice(y, size=len(y), replace=True)
    diffs.append(test_sample.mean() - control_sample.mean())
#границы 95% доверительного интервала
lo, hi = np.percentile(diffs, [2.5, 97.5])
answer_9 = [round(float(lo), 2), round(float(hi), 2)]

answer_9

### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [ ]:
# Задача 10. ρ² с spend (в %) для шести кандидатов из предпериода и отдельно для purchases.
# Ответ — лучший кандидат из предпериода и его ρ² в процентах.

#6 кандидатов в ковариаты, измеренных до эксперимента
candidates = {
    "spend_prev": mine["spend_prev"],
    "spend_prev2": mine["spend_prev2"],
    "spend_prev+spend_prev2": mine["spend_prev"] + mine["spend_prev2"],
    "purchases_prev": mine["purchases_prev"],
    "sessions_prev": mine["sessions_prev"],
    "sessions_prev2": mine["sessions_prev2"],}
#квадрат корреляции с оборотом во время эксперимента, в процентах
r2 = {
    name: mine["spend"].corr(values) ** 2 * 100
    for name, values in candidates.items()}
#для сравнения- покупки во время эксперимента
r2_purchases = mine["spend"].corr(mine["purchases"]) ** 2 * 100
print("Ковариаты до эксперимента:")
for name, value in r2.items():
    print(f"{name}: {value:.1f}%")
print(f"\npurchases во время эксперимента: {r2_purchases:.1f}%")
#выбираем лучшую ковариату из 6
best = max(r2, key=r2.get)
answer_10 = [best, round(float(r2[best]), 1)]

answer_10

### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
# Задача 11. Эффект на spend мэтчингом: Matching(n_neighbors=5),
# признаки age, tenure_days, kyc_level, members, spend_prev, sessions_prev.
# Ответ — ATE, нижняя и верхняя граница интервала.

#создаем числовой индикатор группы: 1-тест, 0-контроль
matching_data = mine.copy()
matching_data["t"] = (matching_data["grp"] == "test").astype(int)
#признаки, измеренные до начала эксперимента
features = ["age", "tenure_days", "kyc_level",
            "members", "spend_prev", "sessions_prev"]
#задаем роли колонок для HypEx
roles = {
    "client_id": InfoRole(str),
    "t": TreatmentRole(),
    "spend": TargetRole(),
    **{f: FeatureRole() for f in features}}
#подбираем по 5 похожих клиентов из другой группы
res_matching = Matching(n_neighbors=5).execute(
    Dataset(roles=roles, data=matching_data, default_role=InfoRole()))
#получаем таблицу ATT, ATC и ATE
matching_table = F(res_matching.resume)
print(matching_table)
#берем ATE и границы его доверительного интервала
ate = matching_table.loc["ATE"]
answer_11 = [
    round(float(ate["Effect Size"]), 2),
    round(float(ate["CI Lower"]), 2),
    round(float(ate["CI Upper"]), 2)]

answer_11

### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [ ]:
# Задача 12. Результат акции в рублях на клиента за 4 недели.
# Допущения — не меняйте их.
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнёрских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, ₽
CHURN_COST = 750      # потеря от ушедшего подписчика: три месяца «Базового», ₽

#доп доход от оборота после CUPED
revenue = effect_cuped["diff"] * COMMISSION
#доп расходы на поддержку
ticket_expense = guards.loc["tickets", "diff"] * TICKET_COST
#потери из-за дополнительного оттока
churn_expense = guards.loc["churned", "diff"] * CHURN_COST
#итоговая выгода на одного клиента за 4 недели
profit = revenue - ticket_expense - churn_expense
print(f"Комиссия с прироста оборота: {revenue:.2f} ₽")
print(f"Расходы на обращения: {ticket_expense:.2f} ₽")
print(f"Потери от оттока: {churn_expense:.2f} ₽")
print(f"Итог на клиента: {profit:.2f} ₽")
answer_12 = round(float(profit), 2)

answer_12

**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [ ]:
# Числа для ответа на вопрос 3 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.

#сравниваем ширину доверительных интервалов 4 методов
widths = {
    "Задача 3 — обычный ДИ": effect["hi"] - effect["lo"],
    "Задача 7 — CUPED": effect_cuped["hi"] - effect_cuped["lo"],
    "Задача 9 — бутстреп": answer_9[1] - answer_9[0],
    "Задача 11 — Matching": answer_11[2] - answer_11[1],}
for method, width in widths.items():
    print(f"{method}: {width:.2f} ₽")
print(f"\nСужение интервала после CUPED: {narrowing:.1f}%")
print(f"Оценка эффекта после CUPED: {effect_cuped['diff']:.2f} ₽")
print(f"Лучшая ковариата: {answer_10[0]}, R² = {answer_10[1]:.1f}%")
print(f"Экономический результат: {answer_12:.2f} ₽ на клиента")

#### ✍️ Ответ на вопрос 3

_Больше всего я бы доверяла оценке CUPED (+132.04 ₽) тк она учитывает оборот клиентов до эксперимента и дает самый узкий доверительный интервал- 274.14 ₽ против 346.61 ₽ у обычного ДИ, 341.83 ₽ у бутстрепа и 286.70 ₽ у Matching. Хотя у purchases самая высокая корреляция с оборотом (ρ² = 70.6%), использовать ее для CUPED нельзя, тк покупки измерялись во время эксперимента и акция могла на них повлиять. По упрощенному расчету акция приносит +1.19 ₽ на клиента, но в нем не учтена стоимость предоставления 2 месяца подписки за 1 ₽. Еще доверительный интервал CUPED включает 0 (от -5.03 до 269.11), поэтому пока нельзя уверенно сказать, что акция увеличивает оборот и действительно окупается_

## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

#### ✍️ Отчёт продакту

**Что проверяли:** Провели A/B-тест акции «Второй месяц Расширенного за 1 ₽» на 6000 клиентах в течение 4 недель. Проверяли, увеличится ли средний оборот в партнерских сервисах минимум на 3% (около 94 ₽ на клиента). Еще смотрели, не вырастут ли отток и количество обращений в поддержку. Проверка распределения клиентов не выявила проблем: SRM p = 0,339

**Что получилось:** После CUPED средний оборот в тестовой группе стал выше на 132,04 ₽ на клиента. Но 95% доверительный интервал от −5,03 до 269,11 ₽, а p = 0,059, поэтому статистически значимый рост подтвердить пока нельзя. Защитные метрики существенно не ухудшились: отток вырос на 0,08 п.п., обращения примерно на 0,0014 на клиента. После поправки Холма ни одна из 4 метрик не осталась статистически значимой

**Какие остались вопросы:** По упрощенной модели акция приносит +1,19 ₽ на клиента за 4 недели: 1,98 ₽ дополнительной комиссии за вычетом 0,20 ₽ расходов на обращения и 0,59 ₽ потерь от оттока. Но здесь не учтена стоимость предоставления 2 месяца подписки за 1 ₽, поэтому реальная прибыль может оказаться отрицательной. Также эксперимент недостаточно мощный для уверенного обнаружения целевого прироста в 94 ₽: расчетный MDE даже с CUPED составляет 186 ₽

**Что рекомендуем:** Пока не запускать акцию на всех клиентов. Сначала нужно оценить полную экономику скидки и при необходимости доработать условия предложения и потом провести повторный эксперимент с большей выборкой. Вернуться к масштабированию можно, когда будет достаточно убедительных данных о росте оборота, приемлемых защитных метриках и положительной экономике акции

## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [ ]:
# Исследование А: ваш код. Всё — от mine.

#1-создаем наблюдательные данные
#акцию увидели только клиенты теста с активностью не ниже медианы
median_sessions = mine["sessions_prev"].median()
active = mine["sessions_prev"] >= median_sessions
obs = mine[
    (mine["grp"] == "control") |
    ((mine["grp"] == "test") & active)].copy()
#HypEx использует числовое обозначение групп
obs["t"] = (obs["grp"] == "test").astype(int)
print(f"Медиана сессий: {median_sessions}")
print("Размеры наблюдательных групп:")
print(obs["grp"].value_counts())

#2-наивная оценка- активные из теста против всего контроля
naive = (
    obs.loc[obs["t"] == 1, "spend"].mean()
    - obs.loc[obs["t"] == 0, "spend"].mean())
print(f"\nНаивная оценка: {naive:.2f} ₽")

#3-эталон из рандомизированного эксперимента
#сравниваем активных клиентов в ОБЕИХ исходных группах
reference = (
    mine.loc[(mine["grp"] == "test") & active, "spend"].mean()
    - mine.loc[(mine["grp"] == "control") & active, "spend"].mean())
print(f"Эталон: {reference:.2f} ₽")
print(f"Смещение наивной оценки: {naive - reference:.2f} ₽")

#4-функция для оценки эффекта через Matching
def matching_att(data, feature_names):
    roles = {
        "client_id": InfoRole(str),
        "t": TreatmentRole(),
        "spend": TargetRole(),
        **{f: FeatureRole() for f in feature_names}}
    result = Matching(n_neighbors=5).execute(
        Dataset(
            roles=roles,
            data=data.copy(),
            default_role=InfoRole()))
    #ATT-эффект для клиентов которые увидели акцию
    return F(result.resume).loc["ATT"]
#признаки клиентов до эксперимента
features_a = [
    "age", "tenure_days", "kyc_level",
    "members", "spend_prev", "sessions_prev"]

#5-Matching с учетом активности
match_full = matching_att(obs, features_a)
print("\nMatching со всеми признаками:")
print(match_full)
print(f"Смещение: {match_full['Effect Size'] - reference:.2f} ₽")

#6-убираем sessions_prev и повторяем Matching
features_without_sessions = [
    f for f in features_a if f != "sessions_prev"]
match_without = matching_att(obs, features_without_sessions)
print("\nMatching без sessions_prev:")
print(match_without)
print(f"Смещение: {match_without['Effect Size'] - reference:.2f} ₽")

#7-сводное сравнение всех оценок
comparison_a = pd.DataFrame({
    "Метод": [
        "Эталон (рандомизация)",
        "Наивная оценка",
        "Matching со всеми признаками",
        "Matching без sessions_prev"],
    "Эффект, ₽": [
        reference,
        naive,
        match_full["Effect Size"],
        match_without["Effect Size"]]})
comparison_a["Отклонение от эталона, ₽"] = (
    comparison_a["Эффект, ₽"] - reference)
print("\nИтоговое сравнение:")
print(comparison_a.round(2).to_string(index=False))

#### ✍️ Исследование А

_Вопрос: хотели проверить, насколько отсутствие рандомизации и отбор только активных клиентов искажают оценку эффекта акции и помогает ли Matching уменьшить это смещение_

_Как проверяли: из тестовой группы оставили только клиентов, у которых число сессий до эксперимента было не ниже медианы, а контрольную группу оставили целиком. Сначала рассчитали обычную разницу средних, потом сравнили ее с эталоном-разницей между активными клиентами исходных тестовой и контрольной групп. Потом применили Matching с 5 соседями и повторили расчет без признака sessions prev_

_Что получилось: наивная оценка составила +1628,47 ₽, хотя эталонный эффект всего +260,05 ₽, то есть завышение составило 1368,42 ₽. Matching со всеми признаками дал +183,83 ₽, что ближе к эталону (отклонение −76,22 ₽). После исключения sessions prev оценка выросла до +637,69 ₽, а отклонение от эталона до 377,64 ₽_

_Что это значит и чего расчёт не доказывает: наивное сравнение сильно завысило эффект, потому что в тесте остались более активные клиенты, а в контроле все подряд. Matching заметно уменьшил смещение, но только когда учитывалась активность до эксперимента. Без sessions prev результат стал менее точным относительно эталона. Matching не гарантирует отсутствие смещения: он учитывает только известные характеристики клиентов и не позволяет полностью исключить влияние ненаблюдаемых факторов_

### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [ ]:
# Исследование Б: ваш код. Всё — от mine.

#1-берем данные контрольной группы где воздействия не было
ctrl = mine[mine["grp"] == "control"].copy()
y = ctrl["spend"].to_numpy()
#эффект который хотим научиться обнаруживать- +3% к обороту
target_effect = 0.03 * y.mean()

#2-считаем CUPED на контрольной группе
#используем оборот до эксперимента как ковариату
#используем функцию CUPED из нашего модуля
y_cuped_series, theta_b = cuped(ctrl["spend"], ctrl["spend_prev"])
y_cuped = y_cuped_series.to_numpy()
print(f"Средний оборот контроля: {y.mean():.2f} ₽")
print(f"Эффект +3%: {target_effect:.2f} ₽")
print(f"Theta CUPED: {theta_b:.3f}")

#3-задаем размеры групп и число повторов
sizes_b = [1000, 3000, 10000, 20000, 40000]
repeats = 500
rng = np.random.default_rng(VARIANT)
results_b = []

#4-для каждого размера группы моделируем 500 экспериментов
for n in sizes_b:
    hits_raw = 0
    hits_cuped = 0
    for _ in range(repeats):
        #выбираем 2n клиентов с возвращением
        #одни и те же индексы используем для обычного анализа и CUPED
        idx = rng.choice(len(y), size=2 * n, replace=True)
        #обычный A/B-тест- 1 группе добавляем эффект
        a = y[idx[:n]] + target_effect
        b = y[idx[n:]]
        p_raw = stats.ttest_ind(a, b, equal_var=False).pvalue
        if p_raw < 0.05:
            hits_raw += 1
        #A/B-тест с CUPED- используем заранее скорректированную метрику
        a_cuped = y_cuped[idx[:n]] + target_effect
        b_cuped = y_cuped[idx[n:]]
        p_cuped = stats.ttest_ind(a_cuped, b_cuped, equal_var=False).pvalue
        if p_cuped < 0.05:
            hits_cuped += 1
    #мощность-доля экспериментов с p < 0,05
    results_b.append({
        "n на группу": n,
        "Без CUPED": hits_raw / repeats,
        "С CUPED": hits_cuped / repeats})

#5-сводная таблица мощности
power_table = pd.DataFrame(results_b)
print("\nМощность A/B-теста:")
print(power_table.round(3).to_string(index=False))

#6-находим первый проверенный размер группы с мощностью >= 80%
for method in ["Без CUPED", "С CUPED"]:
    suitable = power_table.loc[power_table[method] >= 0.8, "n на группу"]
    if len(suitable):
        print(f"{method}: 80% достигается при n = {int(suitable.iloc[0])}")
    else:
        print(f"{method}: 80% не достигнуто на проверенных размерах")

#7-сравниваем с формулой из семинара 7
n_formula_raw = n_per_group(y.std(ddof=1), target_effect)
n_formula_cuped = n_per_group(y_cuped.std(ddof=1), target_effect)
print("\nРасчёт по формуле:")
print(f"Без CUPED: {n_formula_raw} клиентов на группу")
print(f"С CUPED: {n_formula_cuped} клиентов на группу")

#проверяем хватало ли исходных 3000 клиентов на группу
print(f"\nФактически в эксперименте: {len(mine) // 2} на группу")

#### ✍️ Исследование Б

_Вопрос: хотели проверить, сколько клиентов нужно для обнаружения роста оборота на 3% и насколько CUPED помогает уменьшить необходимую выборку. Также сравнили результаты симуляции с формулой расчета мощности_

_Как проверяли: взяли данные контрольной группы, где не было воздействия, и задали эффект +3% от среднего оборота (98,43 ₽). Для каждого размера группы (1000, 3000, 10000, 20000 и 40000) провели по 500 симуляций A/B-теста. Посчитали долю случаев, когда p-value оказался меньше 0,05 отдельно для обычного анализа и CUPED_

_Что получилось: при 3000 клиентах на группу мощность составила всего 22,6% без CUPED и 33% с CUPED. При 10000 клиентах она выросла до 60% и 76% , а при 20000 превысила 80% в обоих случаях. По формуле необходимо 17 642 клиента на группу без CUPED и 11 181 с CUPED. Таким образом, CUPED позволяет довольно сильно уменьшить необходимую выборку_

_Что это значит и чего расчёт не доказывает: наших 3000 клиентов на группу было недостаточно для уверенного обнаружения роста оборота на 3%. Симуляция показывает, что CUPED повышает мощность, но точный минимальный размер выборки по ней определить нельзя тк мы проверяли только 5 значений. Расхождение с формулой связано с крупными шагами между размерами групп, случайностью 500 повторов и особенностями распределения оборота с тяжелым хвостом, поэтому для более точного определения необходимой выборки можно проверить доп размеры групп около 11–18 тысяч клиентов_